# 03 — The Gaussian likelihood, computed safely

*Module 02 · Naive Bayes — notebook 3 of 5*

Notebooks 1 and 2 built likelihoods by **binning** a feature and counting. That worked, but bins are
blunt: the edges are arbitrary, detail inside a bin is lost, and an empty bin gave us that
over-confident zero. This notebook replaces the histogram with a **smooth curve** — a Gaussian fitted
to each class — and in doing so quietly builds the exact model scikit-learn calls `GaussianNB`. Along
the way we meet our first **continuous density**, and the one numerical habit every real Naive Bayes
relies on: working in **log-space**.

**Prerequisites:** notebook 01 (likelihood by binning, the posterior); notebook 02 (the naive product
of per-feature likelihoods).

**What you will be able to do**

- Explain the difference between a probability **mass** and a probability **density**.
- Fit a **Gaussian** likelihood to each class by hand and use it to classify.
- Recognise that by-hand Gaussian naive Bayes **is** `GaussianNB`.
- Explain why multiplying many likelihoods **underflows**, and how **log-space** fixes it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from sklearn.naive_bayes import GaussianNB

from ml_course import datasets, viz
from ml_course.colors import CLASS_CYCLE, COLORS

np.random.seed(0)
viz.use_course_style()

df = datasets.load_penguins()
X, y = datasets.penguins_xy(df)
print(f"rows: {df.shape[0]}   features: {list(X.columns)}   classes: {sorted(y.unique())}")

## A recap, and the trouble with bins

To get a likelihood P(feature | class) in notebook 1, we cut the bill length into bins and counted. It
worked, but look at what it cost us: the bin **edges** were arbitrary (why 42, not 41?), all the detail
*inside* a bin was thrown away, and a bin a class never landed in gave likelihood **zero** — the
over-confident zero-frequency trap. What if, instead of bars, we drew a single **smooth curve** through
each class's measurements?

## First contact: a probability density

Until now every distribution we drew was a **count** — how many penguins fall in a bin. That is a
probability **mass**. But a continuous measurement like bill length has no "how many at *exactly*
45.8 mm" — almost no two birds share an exact value, so any single exact value has essentially zero
probability. The fix is a **density**: a smooth curve where it is the **area** under the curve, not its
height, that gives a probability. The whole curve integrates to **1**, and — because it measures area,
not probability directly — its height can even rise **above 1**. Let us redraw our histogram as a
density and see.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for i, sp in enumerate(sorted(y.unique())):
    values = df.loc[df["species"] == sp, "bill_length_mm"]
    ax.hist(values, bins=15, density=True, color=CLASS_CYCLE[i],
            alpha=0.55, edgecolor="none", label=sp)
ax.set_xlabel("bill_length_mm")
ax.set_ylabel("probability density")
ax.set_title("A density histogram: the bars now integrate to 1")
ax.legend()
plt.show()

**Read the figure.** Same bills as notebook 1, but the y-axis has changed: it is now **density**, and
each species' bars **integrate to 1** (height × width, summed, equals one). A single bar's height
(around 0.14 at most here) is not a probability — the **area** of a stretch of bars is. This is the
object a Gaussian will approximate.

## The Gaussian: one smooth curve per class

The **Gaussian** (the bell curve) is the natural smooth density for a measurement that clusters around a
typical value. It needs only two numbers per class: the **mean** $\mu$ (where the bell sits) and the
**standard deviation** $\sigma$ (how wide it is — the typical distance of a value from the mean).
Modelling each class's feature as a Gaussian,

$$ P(\text{bill} \mid \text{species}) = \text{Gaussian}(\text{bill};\, \mu_{\text{species}},\, \sigma_{\text{species}}) $$

and fitting it is nothing more than computing each class's mean and standard deviation — by hand. This
choice of likelihood is what makes it **Gaussian** naive Bayes.

In [ ]:
xs = np.linspace(df["bill_length_mm"].min() - 2, df["bill_length_mm"].max() + 2, 300)

fig, ax = plt.subplots(figsize=(7, 5))
for i, sp in enumerate(sorted(y.unique())):
    values = df.loc[df["species"] == sp, "bill_length_mm"]
    mu, sigma = values.mean(), values.std(ddof=0)   # ddof=0 matches sklearn's GaussianNB
    print(f"{sp:7}: mu = {mu:.2f} mm, sigma = {sigma:.2f} mm")
    ax.hist(values, bins=15, density=True, color=CLASS_CYCLE[i], alpha=0.40, edgecolor="none",
            label=f"{sp} (histogram)")
    ax.plot(xs, norm.pdf(xs, mu, sigma), color=CLASS_CYCLE[i], linewidth=2.2,
            label=f"{sp} (Gaussian fit)")
ax.set_xlabel("bill_length_mm")
ax.set_ylabel("probability density")
ax.set_title("From histogram (mass) to a smooth Gaussian (density)")
ax.legend(fontsize=9)
plt.show()

**Read the figure.** Each bell sits on its class's mean (Adélie near 38.8 mm, Gentoo near 47.5 mm)
and its width matches the spread. One smooth curve now stands in for the whole histogram — no arbitrary
edges, and crucially **no empty-bin zeros**: a Gaussian assigns a small but non-zero density everywhere,
so the zero-frequency trap of notebook 1 dissolves. Where the bells overlap (around 43 mm) is exactly
where the two species are genuinely hard to tell apart.

## Two features: Gaussian naive Bayes

Notebook 2's naive assumption lets us combine features by **multiplying** their likelihoods. With
Gaussians in hand, the recipe for two features is

$$ P(\text{bill}, \text{flipper} \mid y) \approx \text{Gaussian}(\text{bill} \mid y)\,\cdot\,\text{Gaussian}(\text{flipper} \mid y), $$

and we predict the species with the largest **prior × this product**. That is the whole of Gaussian
naive Bayes. Let us build it by hand and check it against scikit-learn.

In [ ]:
class GaussianNBByHand:
    """Gaussian naive Bayes, fit by hand: one Gaussian per (class, feature)."""

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y)
        self.classes_ = np.array(sorted(np.unique(y)))
        self.priors_, self.mu_, self.sigma_ = {}, {}, {}
        for c in self.classes_:
            Xc = X[y == c]
            self.priors_[c] = len(Xc) / len(X)
            self.mu_[c] = Xc.mean(axis=0)
            self.sigma_[c] = Xc.std(axis=0)          # ddof=0, as GaussianNB
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        # log-posterior (up to the shared evidence) = log prior + sum of per-feature log-densities.
        log_post = np.column_stack([
            np.log(self.priors_[c]) + norm.logpdf(X, self.mu_[c], self.sigma_[c]).sum(axis=1)
            for c in self.classes_
        ])
        return self.classes_[log_post.argmax(axis=1)]


by_hand = GaussianNBByHand().fit(X, y)
sklearn_nb = GaussianNB().fit(X, y)
pred_byhand, pred_sklearn = by_hand.predict(X), sklearn_nb.predict(X)
print(f"by-hand vs sklearn GaussianNB: {(pred_byhand == pred_sklearn).mean() * 100:.1f}% identical")
print(f"by-hand train accuracy: {(pred_byhand == y.to_numpy()).mean():.4f}")

fig = viz.plot_decision_boundary(by_hand, X, y)
fig.axes[0].set_title("By-hand Gaussian naive Bayes — decision boundary")
plt.show()

**Read the figure.** The boundary is gently **curved**, not a straight line — because each class
has its own spread (σ differs by class and feature), the regions bend. And the printout is the
punchline: our by-hand model agrees with scikit-learn's `GaussianNB` on **every** penguin. We did not
approximate the library — we **built** it: by-hand Gaussian NB *is* `GaussianNB` with its
variance-smoothing turned off. (Their internal scores differ by less than a millionth here — far less
than the margin between the classes — so every prediction matches. Notebook 4 turns that dial,
`var_smoothing`, up and watches predictions move.)

## The likelihood is a modelling choice

A Gaussian is the right density for a continuous measurement that clusters around a mean. It is not the
only choice. For **counts** — how many times each word appears in a document — the natural likelihood is
the **multinomial**; for **presence or absence** of each word, the **Bernoulli**. Same Bayes' rule, same
naive product, a different shape for P(feature | class). We build the multinomial on real text in
notebook 5.

## The numerical snag: tiny numbers vanish

A Gaussian density is usually a small number — around 0.1 at a bill's typical value. Multiply two of
them and the product is fine. But Naive Bayes multiplies **one likelihood per feature**, and in notebook
5 a single document has *thousands* of word-features. Multiply thousands of numbers near 0.1 and the
product becomes so small the computer can no longer store it — it rounds to exactly **0.0**, and every
class's score collapses to zero together. Watch it happen.

In [ ]:
ns = np.arange(1, 601)
true_log_prob = ns * np.log(0.1)                       # the honest log-probability (always finite)
products = np.array([np.prod(np.full(n, 0.1)) for n in ns])
underflow_n = int(ns[products == 0.0][0])
print(f"the raw product of {underflow_n} small likelihoods underflows to exactly 0.0")
print(f"  ... yet the sum of their logs is {underflow_n * np.log(0.1):.1f} — perfectly finite")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(ns, true_log_prob, color=COLORS["model"], linewidth=2,
        label="log-probability (sum of logs)")
ax.axvline(
    underflow_n, color=COLORS["error"], linestyle="--", linewidth=1.5,
    label=f"raw float64 product hits 0.0 (N = {underflow_n})",
)
ax.set_xlabel("number of small likelihoods multiplied")
ax.set_ylabel("log-probability")
ax.set_title("Why we work in log-space: the raw product underflows")
ax.legend()
plt.show()

**Read the figure.** The straight line is the true log-probability — it keeps sliding down
steadily, no trouble. The dashed line marks where the **raw product** of those small numbers hits
exactly **0.0** in
float64 (around 324 factors). The product is not *wrong* there; it is **unrepresentable** — too small
for the number format. Taking **logarithms** turns the fragile product into a safe **sum** of
log-likelihoods, and because the logarithm only ever increases, the **argmax is unchanged** — we pick
the same class. And where the raw products have all collapsed to the same 0.0 (an unbreakable tie), the
log-sums stay finite and distinct, so the comparison still works. This is why every real Naive Bayes
adds log-likelihoods instead of multiplying likelihoods (our by-hand `predict` above already does).

## Your turn

1. **Another feature.** Fit a Gaussian to a third penguin measurement (try `bill_depth_mm` from
   `datasets.load_penguins_full()` — this fuller table carries missing values and a third species, so
   drop the NaNs first) and overlay it on that feature's density histogram. Does the bell fit, or does
   the data lean to one side?
2. **A log-posterior by hand.** Pick one penguin. Compute, for each species, log(prior) plus the sum of
   `norm.logpdf` over its two features, and take the argmax. Does it match the by-hand model?
3. **(Harder)** Name a feature where a single Gaussian would fit badly — skewed, or with two humps — and
   say what you would do about it (a transform, or a different likelihood).

## What you built

You retired the histogram. You met a continuous **density** (area, not height, is the probability),
fitted a **Gaussian** likelihood to each class with nothing but a mean and a standard deviation, and
assembled **Gaussian naive Bayes** by hand — confirming it matches scikit-learn's `GaussianNB` exactly.
Then you saw why a product of many likelihoods **underflows** to zero, and adopted the habit that saves
it: add **log-likelihoods** instead of multiplying.

**Vocabulary**

- **probability density** — a smooth curve whose *area* (not height) is a probability; integrates to 1.
- **Gaussian likelihood** — modelling P(feature | class) as a bell with the class's mean and std.
- **Gaussian naive Bayes** — naive Bayes with a Gaussian likelihood per feature; scikit-learn's
  `GaussianNB`.
- **log-space / underflow** — a product of many small numbers rounds to 0.0; summing their logs is safe
  and keeps the same argmax.

## Going further (optional)

- **The formula.** The Gaussian density is P(x) = exp(−(x − μ)² / 2σ²) / (σ·√(2π)). Its logarithm —
  which is what we actually sum — is the tidy −(x − μ)² / 2σ² − log(σ·√(2π)).
- **Is the feature really Gaussian?** A quick check is a QQ-plot (or by eyeing the bell against the
  histogram, as we did). When the fit is poor, a transform or a different likelihood helps.
- **A teaser for notebook 4.** Real `GaussianNB` adds a tiny amount to every variance, `var_smoothing`,
  so a near-constant feature cannot blow up. That dial, and the others, are next.

## References

- G. James, D. Witten, T. Hastie, R. Tibshirani (2021). *An Introduction to Statistical Learning*
  (§4.4.4, naive Bayes). DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).
- T. Hastie, R. Tibshirani, J. Friedman (2009). *The Elements of Statistical Learning* (§6.6.3). DOI:
  [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).

---

*Previous: 02 — The "naive" assumption* · *Next: 04 — The estimators & their parameters*